In [ ]:
from huggingface_hub import HfApi


In [ ]:
from huggingface_hub import HfApi

def main():
    api = HfApi()

    # Trae un bloque pequeño pero suficiente
    models = list(api.list_models(
        limit=1000,
        cardData=True,
        full=False,
        fetch_config=False,
    ))

    selected = []

    for model in models:
        card = getattr(model, "cardData", {}) or {}
        emissions = card.get("co2_eq_emissions")

        if emissions is not None:
            selected.append({
                "modelId": getattr(model, "id", getattr(model, "modelId", None)),
                "emissions": emissions,
                "downloads": getattr(model, "downloads", None),
                "likes": getattr(model, "likes", None),
            })

        if len(selected) == 10:
            break

    print(f"Modelos encontrados: {len(selected)}")
    for row in selected:
        print(row)

if __name__ == "__main__":
    main()

In [ ]:
pip install datasets pyarrow pandas

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

for pkg in ["huggingface_hub", "filelock", "datasets", "requests"]:
    try:
        print(pkg, version(pkg))
    except PackageNotFoundError:
        print(pkg, "no instalado")

print("python:", sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade "filelock>=3.10.0" "requests>=2.32.2"

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "filelock>=3.10.0",
    "requests>=2.32.2",
])

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "datasets",
    "huggingface_hub",
])

In [ ]:
from datasets import load_dataset
import json
import pandas as pd

REPO_ID = "hfmlsoc/hub_weekly_snapshots"
DATE = "2025-01-01"   # reemplázalo por la fecha del snapshot que vayas a usar

def parse_carddata(card):
    if card is None:
        return None
    if isinstance(card, dict):
        return card
    if isinstance(card, str):
        try:
            return json.loads(card)
        except Exception:
            return None
    return None

def normalize_emissions(row):
    card = parse_carddata(row.get("cardData"))
    if not card:
        return None

    emissions = card.get("co2_eq_emissions")
    if emissions is None:
        return None

    base = {
        "modelId": row.get("id"),
        "downloads": row.get("downloads"),
        "likes": row.get("likes"),
        "lastModified": row.get("lastModified"),
        "createdAt": row.get("createdAt"),
        "tags": row.get("tags"),
    }

    if isinstance(emissions, dict):
        base.update({
            "emissions": emissions.get("emissions"),
            "source": emissions.get("source"),
            "training_type": emissions.get("training_type"),
            "geographical_location": emissions.get("geographical_location"),
            "hardware_used": emissions.get("hardware_used"),
        })
    else:
        base.update({
            "emissions": emissions,
            "source": None,
            "training_type": None,
            "geographical_location": None,
            "hardware_used": None,
        })

    return base

def main():
    split_name = DATE.replace("-", "")
    ds = load_dataset(
        REPO_ID,
        data_files={split_name: f"models/{DATE}/models.parquet"}
    )[split_name]

    rows = []
    for row in ds:
        parsed = normalize_emissions(row)
        if parsed is not None:
            rows.append(parsed)

    df = pd.DataFrame(rows)
    print(df.head())
    print("Total modelos con CO2 reportado:", len(df))

    df.to_parquet(f"hf_models_with_co2_{DATE}.parquet", index=False)
    df.to_csv(f"hf_models_with_co2_{DATE}.csv", index=False)

if __name__ == "__main__":
    main()

In [1]:
import sys
import filelock
import requests
import huggingface_hub
from importlib.metadata import version

print("python:", sys.executable)
print("filelock version:", version("filelock"))
print("requests version:", version("requests"))
print("huggingface_hub version:", version("huggingface_hub"))

print("filelock file:", filelock.__file__)
print("requests file:", requests.__file__)
print("huggingface_hub file:", huggingface_hub.__file__)

python: /home/juanfgallo/Downloads/replication_package (1)/.venv/bin/python
filelock version: 3.9.0
requests version: 2.28.2
huggingface_hub version: 1.11.0
filelock file: /app/lib/python3.11/site-packages/filelock/__init__.py
requests file: /app/lib/python3.11/site-packages/requests/__init__.py
huggingface_hub file: /home/juanfgallo/Downloads/replication_package (1)/.venv/lib/python3.11/site-packages/huggingface_hub/__init__.py


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "filelock",
])

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "filelock==3.13.1",
    "requests==2.32.3",
])

In [2]:
import pandas as pd
import json

DATE = "2025-01-01"  # cambia esto por la fecha del snapshot que quieras probar
url = f"https://huggingface.co/datasets/hfmlsoc/hub_weekly_snapshots/resolve/main/models/{DATE}/models.parquet"

df = pd.read_parquet(url)
print(df.head())
print(df.columns)
print(df.shape)

                        _id                                  id  \
0  676c000762cee1f3abc3ed5f             deepseek-ai/DeepSeek-V3   
1  675acf59df0f8590ef7a2e54  PowerInfer/SmallThinker-3B-Preview   
2  676bfff64b96c8ead0a1edf0        deepseek-ai/DeepSeek-V3-Base   
3  66aaa908fc35e079a941470d        black-forest-labs/FLUX.1-dev   
4  676ca1388118866906abbd7c                  hexgrad/Kokoro-82M   

              author gated             inference        lastModified  likes  \
0        deepseek-ai  None  library-not-detected 2024-12-30 03:05:56   1407   
1         PowerInfer  None                  cold 2025-01-06 10:25:53    284   
2        deepseek-ai  None  library-not-detected 2024-12-30 03:05:21   1180   
3  black-forest-labs  None                  warm 2024-08-16 14:38:19   7795   
4            hexgrad  None  library-not-detected 2025-01-06 20:52:51    267   

   trendingScore  private                                       sha  \
0          615.0    False  4c1f24cc10a2a1894304c7ab

In [4]:
import requests
import yaml
import re
import pandas as pd

sample_ids = df["modelId"].dropna().head(20).tolist()

def extract_card_metadata(model_id):
    url = f"https://huggingface.co/{model_id}/raw/main/README.md"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return None

    text = r.text

    # Extraer front matter YAML
    if not text.startswith("---"):
        return None

    parts = text.split("---", 2)
    if len(parts) < 3:
        return None

    yaml_block = parts[1]

    try:
        meta = yaml.safe_load(yaml_block)
    except Exception:
        return None

    return meta

rows = []

for model_id in sample_ids:
    meta = extract_card_metadata(model_id)
    if not meta:
        continue

    emissions = meta.get("co2_eq_emissions")
    if emissions is None:
        continue

    rows.append({
        "modelId": model_id,
        "co2_eq_emissions": emissions
    })

result = pd.DataFrame(rows)
print(result)
print("Encontrados con CO2:", len(result))

Empty DataFrame
Columns: []
Index: []
Encontrados con CO2: 0


In [5]:
import requests
import yaml

def extract_card_metadata(model_id):
    url = f"https://huggingface.co/{model_id}/raw/main/README.md"
    r = requests.get(url, timeout=30)
    print("status:", r.status_code)

    if r.status_code != 200:
        print(r.text[:300])
        return None

    text = r.text
    print(text[:500])  # para inspección

    if not text.startswith("---"):
        return None

    parts = text.split("---", 2)
    if len(parts) < 3:
        return None

    yaml_block = parts[1]

    try:
        return yaml.safe_load(yaml_block)
    except Exception as e:
        print("yaml error:", e)
        return None

meta = extract_card_metadata("bigscience/bloom")
print(meta.get("co2_eq_emissions") if meta else None)

status: 200
---
license: bigscience-bloom-rail-1.0
language:
- ak
- ar
- as
- bm
- bn
- ca
- code
- en
- es
- eu
- fon
- fr
- gu
- hi
- id
- ig
- ki
- kn
- lg
- ln
- ml
- mr
- ne
- nso
- ny
- or
- pa
- pt
- rn
- rw
- sn
- st
- sw
- ta
- te
- tn
- ts
- tum
- tw
- ur
- vi
- wo
- xh
- yo
- zh
- zu
programming_language: 
- C
- C++
- C#
- Go
- Java
- JavaScript
- Lua
- PHP
- Python
- Ruby
- Rust
- Scala
- TypeScript
pipeline_tag: text-generation
widget:
- text: 'A "whatpu" is a small, furry animal native to Tanz
{'emissions': 24700000, 'source': 'Estimating the Carbon Footprint of BLOOM, a 176B Parameter Language Model. https://arxiv.org/abs/2211.02001', 'training_type': 'pre-training', 'geographical_location': 'Orsay, France', 'hardware_used': '384 A100 80GB GPUs'}


In [8]:
import pandas as pd
import requests
import yaml
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Ajustes
SAMPLE_SIZE = 2000
MAX_WORKERS = 16
SAVE_EVERY = 100
RANDOM_STATE = 42

sample_ids = (
    df["modelId"]
    .dropna()
    .drop_duplicates()
    .sample(n=min(SAMPLE_SIZE, df["modelId"].dropna().nunique()), random_state=RANDOM_STATE)
    .tolist()
)

session = requests.Session()
session.headers.update({
    "User-Agent": "hf-co2-replication/0.1"
})

def get_emissions(model_id):
    try:
        url = f"https://huggingface.co/{model_id}/raw/main/README.md"
        r = session.get(url, timeout=20)

        if r.status_code != 200:
            return None

        text = r.text
        if not text.startswith("---"):
            return None

        parts = text.split("---", 2)
        if len(parts) < 3:
            return None

        meta = yaml.safe_load(parts[1])
        if not isinstance(meta, dict):
            return None

        emissions = meta.get("co2_eq_emissions")
        if emissions is None:
            return None

        row = {"modelId": model_id}

        if isinstance(emissions, dict):
            row.update({
                "emissions": emissions.get("emissions"),
                "source": emissions.get("source"),
                "training_type": emissions.get("training_type"),
                "geographical_location": emissions.get("geographical_location"),
                "hardware_used": emissions.get("hardware_used"),
            })
        else:
            row.update({
                "emissions": emissions,
                "source": None,
                "training_type": None,
                "geographical_location": None,
                "hardware_used": None,
            })

        return row

    except Exception:
        return None

rows = []
count_done = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(get_emissions, model_id) for model_id in sample_ids]

    for future in as_completed(futures):
        count_done += 1
        row = future.result()

        if row is not None:
            rows.append(row)

        if count_done % SAVE_EVERY == 0:
            temp_df = pd.DataFrame(rows)
            temp_df.to_csv("hf_co2_partial.csv", index=False)
            print(f"Procesados: {count_done} | Encontrados con CO2: {len(rows)}")

result_df = pd.DataFrame(rows)
result_df.to_csv("hf_co2_sample_results.csv", index=False)
result_df.to_parquet("hf_co2_sample_results.parquet", index=False)

print(result_df.head())
print("Total encontrados con CO2:", len(result_df))

Procesados: 100 | Encontrados con CO2: 1
Procesados: 200 | Encontrados con CO2: 2
Procesados: 300 | Encontrados con CO2: 2
Procesados: 400 | Encontrados con CO2: 2
Procesados: 500 | Encontrados con CO2: 2
Procesados: 600 | Encontrados con CO2: 3
Procesados: 700 | Encontrados con CO2: 3
Procesados: 800 | Encontrados con CO2: 3
Procesados: 900 | Encontrados con CO2: 3
Procesados: 1000 | Encontrados con CO2: 3
Procesados: 1100 | Encontrados con CO2: 3
Procesados: 1200 | Encontrados con CO2: 3
Procesados: 1300 | Encontrados con CO2: 3
Procesados: 1400 | Encontrados con CO2: 3
Procesados: 1500 | Encontrados con CO2: 3
Procesados: 1600 | Encontrados con CO2: 3
Procesados: 1700 | Encontrados con CO2: 3
Procesados: 1800 | Encontrados con CO2: 3
Procesados: 1900 | Encontrados con CO2: 3
Procesados: 2000 | Encontrados con CO2: 3


ArrowTypeError: ("Expected bytes, got a 'float' object", 'Conversion failed for column emissions with type object')